# Mature ChP Subclustering, Stress, and Mitochondrial Signature Analysis

This notebook starts after the broad GSE150903 cell-identity analysis. The goal is to isolate mature choroid plexus-like epithelial cells, rerun dimensionality reduction on that subset, and ask whether mature ChP contains finer sub-states such as ciliated/light, mitochondria-rich/dark, and myoepithelial-like populations.

This notebook also scores stress and mitochondrial gene programs. Because the input is a processed SCT-scaled matrix, this analysis uses gene-signature scores rather than raw mitochondrial read percentages.

## Scope Of This Notebook

This notebook focuses on:

1. loading the annotated full dataset from the previous notebook
2. selecting mature ChP-like cells
3. rerunning PCA on only the mature ChP subset
4. using PCs 1-12 for mature ChP neighbors/UMAP/Leiden clustering, following the paper's reported subclustering parameter
5. validating subclusters with mature ChP, ciliated/light, dark/mitochondria-rich, myoepithelial-like, barrier, and transport markers
6. scoring stress and mitochondrial gene signatures
7. saving the mature ChP subset and selected figures

This notebook does not perform GO/pathway enrichment or external human/mouse reference comparison. Those analyses should go in the next notebook.

## Key Assumptions

This notebook expects the previous methods-guided notebook to have saved:

```text
data/processed/methods_guided_reanalysis.h5ad
```

That object should contain:

- the processed expression matrix
- sample metadata in `adata.obs["sample"]`
- broad cluster labels in `adata.obs["leiden"]`
- a cell-type annotation column such as `cell_type_reviewed` or `cell_type_auto`

If your mature ChP labels have a different column name or wording, update the selection cell below before proceeding.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False)

PROJECT_DIR = Path("/Users/Princess/Documents/Manju's Research Portfolio")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURE_DIR = PROJECT_DIR / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FULL_ADATA_PATH = PROCESSED_DIR / "methods_guided_reanalysis.h5ad"
MATURE_ADATA_PATH = PROCESSED_DIR / "mature_chp_subclustered_stress_mito.h5ad"

print("Project directory:", PROJECT_DIR)
print("Full annotated AnnData path:", FULL_ADATA_PATH)
print("Full annotated AnnData exists:", FULL_ADATA_PATH.exists())

## 1. Load Annotated Full Dataset

Load the full annotated AnnData object produced by the methods-guided notebook. If this file does not exist, first run notebook `02_methods_guided_reanalysis.ipynb` through the save step.

In [ ]:
if not FULL_ADATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {FULL_ADATA_PATH}. Run notebook 02 through the save step first."
    )

adata = sc.read_h5ad(FULL_ADATA_PATH)
print(adata)
print("obs columns:")
print(list(adata.obs.columns))

display(adata.obs.head())

## 2. Choose The Mature ChP Annotation Column

The mature ChP subset has to be selected from a broad cell-type annotation. This notebook looks for `cell_type_reviewed` first, then `cell_type_auto`.

Review the value counts before continuing. If the labels do not contain mature ChP-like wording, edit `label_column` and/or `mature_label_pattern` in the next cell.

In [ ]:
preferred_label_columns = ["cell_type_reviewed", "cell_type_auto", "cell_type", "annotation"]
label_column = next((col for col in preferred_label_columns if col in adata.obs.columns), None)

if label_column is None:
    raise KeyError(
        "No cell-type annotation column found. Expected one of: "
        + ", ".join(preferred_label_columns)
    )

print("Using label column:", label_column)
display(adata.obs[label_column].astype(str).value_counts())

## 3. Subset Mature ChP-Like Cells

The paper subclustered the mature ChP epithelial population after broad clustering. Here, we create a smaller AnnData object containing only cells whose annotation suggests mature ChP identity.

The default pattern is intentionally broad: it captures labels containing `Mature ChP epitheliem'. 
The cells from the telencephalon sample that got pulled into the mature ChP subset were also excluded from the subclustered mature ChP epithelial population.

In [ ]:
mature_label_pattern = "Mature ChP epithelium"
chp_sample_pattern = "Choroid Plexus"

mature_label_mask = (
    adata.obs[label_column]
    .astype(str)
    .str.contains(mature_label_pattern, case=False, regex=True, na=False)
)

chp_sample_mask = (
    adata.obs["sample"]
    .astype(str)
    .str.contains(chp_sample_pattern, case=False, regex=True, na=False)
)

mature_mask = mature_label_mask & chp_sample_mask

adata_mature = adata[mature_mask].copy()

print("Mature ChP epithelial cells selected from ChP organoid samples:", adata_mature.n_obs)
print("Genes retained:", adata_mature.n_vars)

print("\nSelected label counts:")
display(adata_mature.obs[label_column].astype(str).value_counts())

print("\nSelected sample counts:")
display(adata_mature.obs["sample"].astype(str).value_counts())

print("\nExcluded mature-label cells from non-ChP samples:")
excluded_mature_non_chp = adata[mature_label_mask & ~chp_sample_mask].copy()
display(excluded_mature_non_chp.obs["sample"].astype(str).value_counts())

if adata_mature.n_obs == 0:
    print("Available labels:")
    print(adata.obs[label_column].astype(str).value_counts())

    print("\nAvailable samples:")
    print(adata.obs["sample"].astype(str).value_counts())

    raise ValueError(
        "No mature ChP epithelial cells selected from ChP samples. "
        "Check mature_label_pattern, label_column, and sample labels before continuing."
    )

## 4. Rerun PCA, Neighbors, UMAP, And Leiden On Mature ChP Cells

The paper reports that mature ChP subclustering used PCs 1-12, selected by ElbowPlot. These PCs are recalculated within the mature ChP subset; they are not reused from the full-dataset PCA.

In [ ]:
sc.tl.pca(adata_mature, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata_mature, log=True, n_pcs=30)

sc.pp.neighbors(
    adata_mature,
    n_neighbors=15,
    n_pcs=12,
)

sc.tl.umap(
    adata_mature,
    min_dist=0.2,
    random_state=42,
)

sc.tl.leiden(
    adata_mature,
    resolution=0.5,
    flavor="igraph",
    n_iterations=2,
    directed=False,
)

plot_colors = ["sample", "leiden"] if "sample" in adata_mature.obs.columns else ["leiden"]
sc.pl.umap(
    adata_mature,
    color=plot_colors,
    size=8,
    alpha=0.85,
    wspace=0.4,
    frameon=False,
    title=["Mature ChP subset by sample", "Mature ChP subclusters"][:len(plot_colors)],
)

print("Mature ChP subclustering complete.")

## 5. Four-Panel Sample Highlight UMAP For Mature ChP Subset

This view asks whether mature ChP-like cells from different samples occupy similar or distinct regions of the mature ChP UMAP. Overlap is biologically meaningful and may indicate shared mature ChP states across timepoints.

In [ ]:
if "sample" not in adata_mature.obs.columns:
    print("No sample column found; skipping sample-highlight plot.")
else:
    umap = adata_mature.obsm["X_umap"]
    sample_series = adata_mature.obs["sample"].astype(str)
    samples = list(adata_mature.obs["sample"].cat.categories) if hasattr(adata_mature.obs["sample"], "cat") else sorted(sample_series.unique())

    sample_colors = {
        "Choroid Plexus Org D27": "#1f77b4",
        "Choroid Plexus Org D46": "#ff7f0e",
        "Choroid Plexus Org D53": "#2ca02c",
        "Telencephalon organoids D55": "#d62728",
    }

    ncols = 2
    nrows = int(np.ceil(len(samples) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(10, 4.5 * nrows), sharex=True, sharey=True)
    axes = np.array(axes).reshape(-1)

    x_pad = (umap[:, 0].max() - umap[:, 0].min()) * 0.05
    y_pad = (umap[:, 1].max() - umap[:, 1].min()) * 0.05
    xlim = (umap[:, 0].min() - x_pad, umap[:, 0].max() + x_pad)
    ylim = (umap[:, 1].min() - y_pad, umap[:, 1].max() + y_pad)

    for ax, sample in zip(axes, samples):
        mask = sample_series.to_numpy() == str(sample)
        color = sample_colors.get(str(sample), "#1f77b4")
        ax.scatter(umap[~mask, 0], umap[~mask, 1], s=2, c="#d0d0d0", alpha=0.2, linewidths=0, rasterized=True)
        ax.scatter(umap[mask, 0], umap[mask, 1], s=4, c=color, alpha=0.9, linewidths=0, rasterized=True)
        ax.set_title(str(sample), fontsize=12)
        ax.set_xlabel("UMAP1")
        ax.set_ylabel("UMAP2")
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(False)

    for ax in axes[len(samples):]:
        ax.set_visible(False)

    fig.suptitle("Mature ChP Sample Distribution", fontsize=16, y=0.98)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "mature_chp_sample_highlight_umap.png", bbox_inches="tight", dpi=300)
    plt.show()

## 6. Validate Mature ChP Subclusters With Marker Sets

This section checks whether mature ChP subclusters show marker patterns consistent with the paper's described ChP epithelial states.

Interpretation guide:

- `TTR`, `AQP1`, `KRT18`, `NME5`: mature ChP epithelial identity
- `FOXJ1`, `ARL13B`, `CCDC67`: light/ciliated ChP-like state
- `CARD19`, `IGF2`, `RBP1`: dark/mitochondria-rich ChP-like state
- `KRT17`, `ACTA2`, `TAGLN`: myoepithelial-like state
- `CLDN1`, `CLDN3`, `TJP1`, `AQP1`, `CA2`, `SLC23A2`: barrier/transport-associated genes

In [ ]:
mature_chp_marker_sets = {
    "Mature ChP epithelium": ["TTR", "AQP1", "KRT18", "NME5"],
    "Light / ciliated ChP": ["FOXJ1", "ARL13B", "CCDC67"],
    "Dark / mitochondria-rich ChP": ["CARD19", "IGF2", "RBP1"],
    "Myoepithelial-like ChP": ["KRT17", "ACTA2", "TAGLN"],
    "Barrier / tight junction": ["CLDN1", "CLDN3", "CLDN5", "TJP1", "TJP2", "OCLN"],
    "CSF secretion / transport": ["AQP1", "CA2", "CA12", "SLC23A2", "SLC46A1"],
    "Cycling": ["MKI67", "TOP2A", "PCNA"],
}

mature_chp_marker_sets_present = {
    group: [gene for gene in genes if gene in adata_mature.var_names]
    for group, genes in mature_chp_marker_sets.items()
}
mature_chp_marker_sets_present = {
    group: genes for group, genes in mature_chp_marker_sets_present.items() if genes
}

print("Marker sets present in mature ChP subset:")
for group, genes in mature_chp_marker_sets_present.items():
    print(f"{group}: {genes}")

# UMAP of mature ChP subset colored by Leiden subclusters
sc.pl.umap(
    adata_mature,
    color="leiden",
    size=8,
    alpha=0.85,
    frameon=False,
    legend_loc="right margin",
    title="Mature ChP subclusters",
    show=False,
)

plt.savefig(
    FIGURE_DIR / "mature_chp_umap_by_leiden.png",
    bbox_inches="tight",
    dpi=300,
)
plt.show()

sc.pl.dotplot(
    adata_mature,
    var_names=mature_chp_marker_sets_present,
    groupby="leiden",
    standard_scale="var",
    dendrogram=False,
)

plt.savefig(FIGURE_DIR / "mature_chp_marker_dotplot_by_leiden.png", bbox_inches="tight", dpi=300)

## 7. Rank Marker Genes Within Mature ChP Subclusters

This ranks genes that distinguish mature ChP subclusters from one another. Because the matrix is SCT-scaled, use these rankings as exploratory marker evidence and validate important genes with marker plots and known biology.

In [ ]:
sc.tl.rank_genes_groups(
    adata_mature,
    groupby="leiden",
    method="wilcoxon",
)

sc.pl.rank_genes_groups(
    adata_mature,
    n_genes=10,
    sharey=False,
)

mature_marker_table = sc.get.rank_genes_groups_df(adata_mature, group=None)
mature_marker_table.to_csv(PROCESSED_DIR / "mature_chp_subcluster_marker_genes.csv", index=False)
display(mature_marker_table.head(30))

## 8. Score Stress, Mitochondrial, And ChP State Gene Programs

This section calculates per-cell scores for biologically meaningful gene sets. Since this is not raw count data, the mitochondrial score here is a mitochondrial gene-expression signature, not a raw mitochondrial read percentage.

In [ ]:
signature_sets = {
    "stress_score": [
        "FOS", "JUN", "JUNB", "JUND", "ATF3", "DDIT3",
        "HSPA1A", "HSPA1B", "HSP90AA1", "DNAJB1",
    ],
    "mitochondrial_gene_score": [
        "MT-CO1", "MT-CO2", "MT-CO3", "MT-ND1", "MT-ND2", "MT-ND3",
        "MT-ND4", "MT-ND5", "MT-CYB", "MT-ATP6", "MT-ATP8",
    ],
    "dark_mito_chp_score": ["CARD19", "IGF2", "RBP1"],
    "light_ciliated_chp_score": ["FOXJ1", "ARL13B", "CCDC67"],
    "myoepithelial_like_score": ["KRT17", "ACTA2", "TAGLN"],
    "barrier_transport_score": ["CLDN1", "CLDN3", "TJP1", "AQP1", "CA2", "SLC23A2"],
}

score_columns = []
for score_name, genes in signature_sets.items():
    genes_present = [gene for gene in genes if gene in adata_mature.var_names]
    print(f"{score_name}: {len(genes_present)} / {len(genes)} genes present -> {genes_present}")
    if len(genes_present) == 0:
        print(f"Skipping {score_name}; no genes present in matrix.")
        continue
    sc.tl.score_genes(
        adata_mature,
        gene_list=genes_present,
        score_name=score_name,
    )
    score_columns.append(score_name)

print("Scores added:", score_columns)

## 9. Visualize Signature Scores On UMAP

These plots help ask whether stress, mitochondrial, or dark/ciliated ChP programs localize to specific mature ChP subclusters.

In [ ]:
if score_columns:
    sc.pl.umap(
        adata_mature,
        color=score_columns,
        cmap="viridis",
        size=8,
        frameon=False,
        ncols=3,
        show=False,
    )

    plt.savefig(
        FIGURE_DIR / "mature_chp_signature_scores_umap.png",
        bbox_inches="tight",
        dpi=300,
    )
    plt.show()

else:
    print("No score columns available to plot.")

## Interpretation: Stress, Mitochondrial, And ChP State Scores

The stress signature is low across most of the mature ChP UMAP, suggesting that this subset is not dominated by a broad stress-associated transcriptional program. This does not prove that no cells are stressed, but the selected stress markers do not define a major region of the embedding.

The mitochondrial gene score is more widely distributed and shows regional enrichment across parts of the mature ChP subset. This indicates variation in mitochondrial gene expression among mature ChP-like cells.

The dark/mitochondria-rich ChP score is more focal than the broader mitochondrial gene score. This suggests that the `CARD19`/`IGF2`/`RBP1` dark ChP-like program may identify a more specific subpopulation within a larger region of cells with elevated mitochondrial gene expression.

The barrier/transport score partially overlaps with areas of elevated mitochondrial gene expression. This may indicate that some metabolically active mature ChP states also express transport/barrier-associated genes, consistent with the secretory and barrier functions of choroid plexus epithelium.

Because these scores are calculated from a processed SCT-scaled matrix, they should be interpreted as relative gene-program scores rather than raw mitochondrial percentage or direct stress measurements. The next step is to summarize these scores by Leiden subcluster and sample to test whether the apparent UMAP patterns are cluster-specific, sample-specific, or broadly distributed.

## 10. Compare Signature Scores Across Subclusters And Samples

These summaries help determine whether a signature is subcluster-specific, sample-specific, or broadly distributed.

In [ ]:
# Summarize and visualize signature scores by Leiden subcluster and sample.

if not score_columns:
    print("No score columns available to summarize. Run the signature-scoring cell first.")

else:
    # Mean scores by mature ChP Leiden subcluster
    cluster_score_summary = (
        adata_mature.obs
        .groupby("leiden", observed=True)[score_columns]
        .mean()
    )

    display(cluster_score_summary)

    cluster_score_summary.to_csv(
        PROCESSED_DIR / "mature_chp_signature_scores_by_leiden.csv"
    )

    ax = cluster_score_summary.plot(
        kind="bar",
        figsize=(12, 5)
    )

    ax.set_ylabel("Mean signature score")
    ax.set_xlabel("Mature ChP Leiden subcluster")
    ax.set_title("Mean Signature Scores By Mature ChP Subcluster")
    plt.xticks(rotation=0)
    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / "mature_chp_signature_scores_by_leiden.png",
        bbox_inches="tight",
        dpi=300
    )

    plt.show()

    # Mean scores by sample
    if "sample" in adata_mature.obs.columns:
        sample_score_summary = (
            adata_mature.obs
            .groupby("sample", observed=True)[score_columns]
            .mean()
        )

        display(sample_score_summary)

        sample_score_summary.to_csv(
            PROCESSED_DIR / "mature_chp_signature_scores_by_sample.csv"
        )

        # Dot plot of mean signature scores by sample
        plot_df = sample_score_summary.copy()

        fig, ax = plt.subplots(figsize=(10, 4.5))

        x_labels = plot_df.columns.tolist()
        y_labels = plot_df.index.astype(str).tolist()

        x = []
        y = []
        colors = []
        sizes = []

        for yi, sample in enumerate(y_labels):
            for xi, score_name in enumerate(x_labels):
                value = plot_df.loc[sample, score_name]
                x.append(xi)
                y.append(yi)
                colors.append(value)
                sizes.append(max(value, 0) * 45 + 40)

        scatter = ax.scatter(
            x,
            y,
            c=colors,
            s=sizes,
            cmap="viridis",
            edgecolor="black",
            linewidth=0.3,
        )

        ax.set_xticks(range(len(x_labels)))
        ax.set_xticklabels(x_labels, rotation=45, ha="right")
        ax.set_yticks(range(len(y_labels)))
        ax.set_yticklabels(y_labels)

        ax.set_xlabel("Signature score")
        ax.set_ylabel("Sample")
        ax.set_title("Mean Signature Scores By Sample")

        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label("Mean signature score")

        plt.tight_layout()

        plt.savefig(
            FIGURE_DIR / "mature_chp_signature_scores_by_sample_dotplot.png",
            bbox_inches="tight",
            dpi=300,
        )

        plt.show()

    else:
        print("No sample column found; skipping sample-level summary.")

## 11. Interpretation Checkpoint

Before moving to enrichment or reference comparison, review:

- Which mature ChP subclusters express `CARD19`, `IGF2`, and `RBP1`?
- Are ciliated/light markers such as `FOXJ1`, `ARL13B`, and `CCDC67` localized to a specific subcluster?
- Are stress signatures concentrated in one subcluster or one sample?
- Does the mitochondrial gene score match the dark/mitochondria-rich ChP marker score, or do they separate?
- Are any subclusters mostly from a single timepoint, suggesting developmental differences?

If the subcluster labels are unclear, adjust Leiden resolution and rerun marker validation before continuing.

## Interpretation Checkpoint: Signature Scores

Based on the mean signature scores by Leiden subcluster, the strongest dark/mitochondria-rich ChP-like signal is seen in subclusters **1**, **0**, **5**, and **2**, which have the highest `dark_mito_chp_score`. These clusters are the best candidates for cells expressing the `CARD19`/`IGF2`/`RBP1` dark ChP-like program. Subclusters **7**, **10**, **6**, and **8** have negative dark ChP scores and are less consistent with that identity.

The light/ciliated ChP score is modest overall. The highest values are in subclusters **0**, **2**, **1**, and **11**, but the score range is small compared with the mitochondrial score. This suggests that the `FOXJ1`/`ARL13B`/`CCDC67` program may be present in a limited or weakly separated subset rather than defining one dominant ciliated cluster.

Stress scores are low and fairly similar across subclusters and samples. The highest subcluster-level stress scores are in **7**, **6**, **12**, and **11**, but the differences are not dramatic. At the sample level, D27 has the highest mean stress score, followed by D46 and D53. Overall, there is no evidence for one strongly stress-dominated mature ChP subcluster.

The mitochondrial gene score and the dark/mitochondria-rich ChP score do not perfectly match. Subcluster **8** has the strongest mitochondrial gene score by far, but it has a negative `dark_mito_chp_score`. In contrast, subclusters **1** and **0** have high dark ChP scores and also elevated mitochondrial scores. This suggests that broad mitochondrial gene expression and the specific dark ChP marker program are related but separable. In other words, not every mitochondria-high cell is necessarily a dark ChP-like cell by `CARD19`/`IGF2`/`RBP1` markers.

At the sample level, D53 has the highest mitochondrial gene score, dark ChP score, light/ciliated score, myoepithelial-like score, and the least negative barrier/transport score. D27 has the lowest mitochondrial and dark ChP scores and the highest stress score. This suggests possible developmental differences across ChP organoid timepoints, with D53 showing stronger mature/dark ChP-associated programs.
The mature ChP sample-composition table should be used together with these score summaries to determine whether score-enriched subclusters are also enriched for specific timepoints.

In [ ]:
# Sample composition of each mature ChP Leiden subcluster.
# This asks whether specific mature ChP subclusters are enriched for D27, D46, or D53 cells.

if "sample" not in adata_mature.obs.columns:
    print("No sample column found in adata_mature.obs.")

else:
    # Raw cell counts
    mature_sample_counts = pd.crosstab(
        adata_mature.obs["leiden"],
        adata_mature.obs["sample"]
    )

    display(mature_sample_counts)

    mature_sample_counts.to_csv(
        PROCESSED_DIR / "mature_chp_sample_counts_by_leiden.csv"
    )

    # Percent of each Leiden subcluster from each sample
    mature_sample_percent = pd.crosstab(
        adata_mature.obs["leiden"],
        adata_mature.obs["sample"],
        normalize="index"
    ) * 100

    mature_sample_percent = mature_sample_percent.round(2)

    display(mature_sample_percent)

    mature_sample_percent.to_csv(
        PROCESSED_DIR / "mature_chp_sample_percent_by_leiden.csv"
    )

    # Heatmap visualization
    fig, ax = plt.subplots(figsize=(9, 6))

    im = ax.imshow(
        mature_sample_percent,
        aspect="auto",
        cmap="viridis"
    )

    ax.set_xticks(range(mature_sample_percent.shape[1]))
    ax.set_xticklabels(
        mature_sample_percent.columns.astype(str),
        rotation=45,
        ha="right"
    )

    ax.set_yticks(range(mature_sample_percent.shape[0]))
    ax.set_yticklabels(mature_sample_percent.index.astype(str))

    ax.set_xlabel("Sample")
    ax.set_ylabel("Mature ChP Leiden subcluster")
    ax.set_title("Sample Composition Of Mature ChP Subclusters")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Percent of subcluster cells")

    # Add percent labels inside heatmap cells
    for i in range(mature_sample_percent.shape[0]):
        for j in range(mature_sample_percent.shape[1]):
            value = mature_sample_percent.iloc[i, j]
            text_color = "white" if value > 50 else "black"
            ax.text(
                j,
                i,
                f"{value:.1f}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=8
            )

    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / "mature_chp_sample_composition_by_leiden_heatmap.png",
        bbox_inches="tight",
        dpi=300
    )

    plt.show()

In [ ]:
cluster_10_markers = mature_marker_table[
    mature_marker_table["group"].astype(str) == "10"
].head(50)

display(cluster_10_markers)

cluster_10_top_genes = cluster_10_markers["names"].head(10).tolist()

sc.pl.dotplot(
    adata_mature,
    var_names=cluster_10_top_genes,
    groupby="leiden",
    standard_scale="var",
    dendrogram=False
)

## Cluster 10 Candidate Markers

Subcluster 10 is enriched in the D27 mature ChP subset and decreases in D46/D53. Its signature-score profile is not strongly dark/mitochondria-rich, ciliated/light, myoepithelial-like, or barrier/transport-high, suggesting it may represent an earlier or less specialized mature ChP-like state.

Marker inspection highlighted `CRABP2` and `SRRM2` among genes of interest. `CRABP2` motivated a retinoic-acid-associated marker check below. `SRRM2` is involved in RNA processing/splicing and has been implicated in neurological and neurodevelopmental disease contexts, so its presence is hypothesis-generating. In this non-disease organoid dataset, `SRRM2` should not be interpreted as disease evidence, but it may point to an RNA-processing-associated early ChP state worth following up in future disease-relevant ChP/barrier datasets.

In [ ]:
# Retinoic-acid-associated signature analysis.
# CRABP2 was observed in the D27-enriched mature ChP subcluster 10.
# This cell tests whether CRABP2 is part of a broader retinoic-acid-associated program.

retinoic_acid_markers = [
    "CRABP2", "CRABP1",
    "RBP1", "RBP2", "RBP4",
    "ALDH1A1", "ALDH1A2", "ALDH1A3",
    "RARRES1", "RARRES2", "RARRES3",
    "STRA6",
    "CYP26A1", "CYP26B1", "CYP26C1",
    "RARA", "RARB", "RARG",
    "RXRA", "RXRB", "RXRG",
]

ra_genes_present = [
    gene for gene in retinoic_acid_markers
    if gene in adata_mature.var_names
]

ra_genes_missing = [
    gene for gene in retinoic_acid_markers
    if gene not in adata_mature.var_names
]

print("Retinoic-acid-associated genes present:")
print(ra_genes_present)

print("\nRetinoic-acid-associated genes missing:")
print(ra_genes_missing)

if len(ra_genes_present) == 0:
    raise ValueError("No retinoic-acid-associated genes found in adata_mature.var_names.")

sc.tl.score_genes(
    adata_mature,
    gene_list=ra_genes_present,
    score_name="retinoic_acid_score"
)

# UMAP visualization
plot_genes = [
    gene for gene in ["CRABP2", "CRABP1", "RBP1", "ALDH1A1", "STRA6"]
    if gene in adata_mature.var_names
]

sc.pl.umap(
    adata_mature,
    color=plot_genes + ["retinoic_acid_score", "leiden", "sample"],
    cmap="viridis",
    size=8,
    frameon=False,
    ncols=3,
)

# Mean retinoic acid score by mature ChP subcluster
ra_by_leiden = (
    adata_mature.obs
    .groupby("leiden", observed=True)[["retinoic_acid_score"]]
    .mean()
    .sort_values("retinoic_acid_score", ascending=False)
)

display(ra_by_leiden)

ra_by_leiden.to_csv(
    PROCESSED_DIR / "mature_chp_retinoic_acid_score_by_leiden.csv"
)

# Mean retinoic acid score by sample
if "sample" in adata_mature.obs.columns:
    ra_by_sample = (
        adata_mature.obs
        .groupby("sample", observed=True)[["retinoic_acid_score"]]
        .mean()
        .sort_values("retinoic_acid_score", ascending=False)
    )

    display(ra_by_sample)

    ra_by_sample.to_csv(
        PROCESSED_DIR / "mature_chp_retinoic_acid_score_by_sample.csv"
    )

# Dotplot for retinoic-acid-associated genes by Leiden cluster
if plot_genes:
    sc.pl.dotplot(
        adata_mature,
        var_names=plot_genes,
        groupby="leiden",
        standard_scale="var",
        dendrogram=False,
    )

## Retinoic-Acid-Associated Marker Check

`CRABP2` was observed among genes elevated in the D27-enriched mature ChP subcluster 10, raising the possibility that this subcluster may reflect an early or developmental ChP state with retinoic-acid-associated features.

To test whether this was part of a broader retinoic-acid-related program, I checked additional genes including `CRABP1`, `RBP1`, `ALDH1A1`, and `STRA6`.

The dotplot suggests that this is **not a strong broad retinoic acid pathway signature** in the mature ChP subset. `RBP1` is broadly expressed across many mature ChP subclusters, while `CRABP2` is more localized but not supported by strong expression of several other retinoic-acid-associated genes. `ALDH1A1` and `STRA6` show weak or limited signal.

Therefore, `CRABP2` should be interpreted as a candidate marker of the D27-enriched subcluster 10 rather than definitive evidence of a full retinoic-acid transcriptional program. This observation is still useful as a hypothesis-generating result and could be followed up in future disease or developmental datasets, including Parkinson's disease-relevant ChP/barrier datasets if available.

## 12. Save Mature ChP Subset

Save the mature ChP subset with subcluster labels and signature scores for the next notebook.

In [ ]:
adata_mature.write_h5ad(MATURE_ADATA_PATH)
print("Saved mature ChP subset:", MATURE_ADATA_PATH)
print(adata_mature)